# epsilon_data cookbook
Short recipes, a few lines each, all through the public API. Pair with `epsilon_data/README.md` and the worked-examples notebook.

In [1]:
import sys, pathlib
root = next((p for p in pathlib.Path.cwd().parents if (p/'epsilon_data').is_dir()), pathlib.Path.cwd().parent)
sys.path.insert(0, str(root))
import pandas as pd, epsilon_data as ed
print('epsilon_data', ed.__version__)

epsilon_data 1.0


### 1 · Filter markets by liquidity

In [2]:
ed.catalog(universe='politics_negrisk', min_trades=200).sort_values('n_trades', ascending=False).head(10)[['question','outcome_label','n_trades','median_spread_cents']]

,question,outcome_label,n_trades,median_spread_cents
761,Will there be no change in Fed interest rates ...,Yes,26419,0.6
764,Will the Fed decrease interest rates by 25 bps...,No,24955,0.2
779,Will Mojtaba Khamenei be head of state in Iran...,Yes,24568,0.5
754,Will the Fed decrease interest rates by 25 bps...,No,14592,0.1
945,Will Josh Stein win the 2028 US Presidential E...,No,14399,0.1
757,Will the Fed increase interest rates by 25 bps...,Yes,13056,0.4
762,Will there be no change in Fed interest rates ...,No,13021,0.6
771,Will there be no change in Fed interest rates ...,Yes,12421,2.0
760,Will the Fed increase interest rates by 50+ bp...,No,12381,0.2
959,Will Pete Buttigieg win the 2028 US Presidenti...,No,12361,0.1


### 2 · Find the busiest hour (UTC) to trade a universe

In [3]:
a = ed.activity_by_time('esports')
a.groupby('hour_of_day')['n_trades'].sum().sort_values(ascending=False).head(5)

hour_of_day
10    407858
13    400697
12    400154
9     386294
15    374446
Name: n_trades, dtype: int64

### 3 · Pull every market in an event

In [4]:
ev = ed.events(universe='politics_negrisk').iloc[0]['event_slug']
ed.catalog(event=ev)[['market_slug','outcome_label','n_trades','median_mid']]

,market_slug,outcome_label,n_trades,median_mid
0,will-abigail-spanberger-win-the-2028-democrati...,Yes,64,0.0015
1,will-abigail-spanberger-win-the-2028-democrati...,No,30,0.9985
2,will-adam-schiff-win-the-2028-democratic-presi...,Yes,17,0.0015
3,will-adam-schiff-win-the-2028-democratic-presi...,No,6,0.9985
4,will-alex-padilla-win-the-2028-democratic-pres...,Yes,20,0.0015
...,...,...,...,...
97,will-tim-walz-win-the-2028-democratic-presiden...,No,5934,0.9960
98,will-wes-moore-win-the-2028-democratic-preside...,Yes,170,0.0640
99,will-wes-moore-win-the-2028-democratic-preside...,No,622,0.9360
100,will-zohran-mamdani-win-the-2028-democratic-pr...,Yes,963,0.0050


### 4 · Compare two markets' spreads

In [5]:
top = ed.catalog(universe='politics_negrisk', min_trades=100).sort_values('n_trades', ascending=False)
top.head(2).set_index('question')[['median_spread_cents','n_trades','median_mid']]

,median_spread_cents,n_trades,median_mid
question,,,
Will there be no change in Fed interest rates after the July 2026 meeting?,0.6,26419,0.7675
Will the Fed decrease interest rates by 25 bps after the September 2026 meeting?,0.2,24955,0.9690


### 5 · Build a returns series from a token's mid

In [6]:
aid = ed.catalog(min_trades=100).sort_values('n_trades', ascending=False).iloc[0]['asset_id']
l1 = ed.load_l1(aid).set_index('ts')
ret = l1['mid'].resample('5min').last().ffill().pct_change().dropna()
ret.describe()

count    710.000000
mean       0.035891
std        0.577336
min       -0.888889
25%        0.000000
50%        0.000000
75%        0.000000
max       12.000000
Name: mid, dtype: float64

### 6 · Slice a token's tape by time window

In [7]:
import pandas as pd
end = l1.index.max(); start = end - pd.Timedelta('6h')
ed.load_l1(aid, start=start, end=end)[['ts','best_bid','best_ask','mid','spread_c']].head()

,ts,best_bid,best_ask,mid,spread_c
0,2026-08-03 06:00:00.002000+00:00,0.003,0.010,0.0065,0.7
1,2026-08-03 06:24:59.927000+00:00,0.003,0.009,0.0060,0.6
2,2026-08-03 07:00:00.134000+00:00,0.003,0.009,0.0060,0.6
3,2026-08-03 08:00:00.600000+00:00,0.003,0.009,0.0060,0.6
4,2026-08-03 09:00:00.327000+00:00,0.003,0.009,0.0060,0.6


### 7 · The YES/NO mirror in one call

In [8]:
cid = ed.catalog(min_trades=100).sort_values('n_trades', ascending=False).iloc[0]['condition_id']
pair = ed.load_pair(cid); (pair[0] + pair[1]).describe()   # ~1 for a healthy market

count    5.030000e+02
mean     1.000000e+00
std      2.477584e-17
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64

### 8 · Adverse-selection markout for a token

In [9]:
mo = ed.markout(aid)
{h: round(mo[f'markout_{h}'].mean()*100, 3) for h in (10,30,60)}   # ¢, maker view; negative = toxic

{10: np.float64(0.05), 30: np.float64(0.049), 60: np.float64(0.049)}

### 9 · NegRisk YES-sum (instantaneous, not a sum of medians)

In [10]:
negev = ed.events(universe='politics_negrisk')
negev = negev[(negev.neg_risk==True) & (negev.n_markets>=4)].iloc[0]['event_slug']
ns = ed.negrisk_sum(negev); ns['yes_sum'].describe()

count    9897.000000
mean        0.981450
std         0.032002
min         0.007500
25%         0.961500
50%         0.982000
75%         1.005000
max         1.048000
Name: yes_sum, dtype: float64

### 10 · Audit a market (report + recommendation; never writes)

In [11]:
r = ed.audit_market(cid)
print(r.verdict, '—', r.reason)
pd.DataFrame([{'check':c.name,'level':c.level,'detail':c.detail} for c in r.checks])

worth a look — 17 gaps >1h (17 not explained by the known outage)


,check,level,detail
0,identity,ok,identity_status=resolved; failed=none; complem...
1,value sanity,ok,no crossed/out-of-range/frozen values; spread=...
2,pair sum≈1,ok,"median |A+B-1|=0.000, worst=0.000 over 503 pts"
3,continuity,note,17 gaps >1h (17 not explained by the known out...
4,trades vs quotes,ok,"0/115890 trades outside touch (>1¢), 0 with no..."
5,volume shape,ok,no dominant print / dup hashes
6,resolution,ok,not resolved / not applicable


### 11 · Coverage — true gaps vs quiet hours

In [12]:
cov = ed.coverage('esports'); cov[cov.n_missing>0][['date','missing_hours']]   # no-book = true gap

,date,missing_hours
0,2026-06-19,"[00, 01, 02, 03, 04, 05, 06, 07, 08, 09, 10, 11]"
3,2026-06-22,"[15, 16, 17, 18, 19, 20, 21, 22, 23]"
4,2026-06-23,"[00, 01, 02, 03, 04, 05, 06, 07]"
63,2026-08-21,"[21, 22, 23]"
